## ΔiAUC

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc


# ============================= Configuration =============================
PROJECT_DIR = Path("./work/")
DATA_PATH = PROJECT_DIR / "UKB_ROC_raw.csv"

OUT_DIR = PROJECT_DIR / "Brain_Vital8_ROC" / "dynAUC_CI_and_Delta"
FIG_DIR = OUT_DIR / "figures"
CSV_DIR = OUT_DIR / "tables"
for d in [OUT_DIR, FIG_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

time_col = "time"
event_col = "status"
group_col = "health_status"

scores = ["BrainVital8", "libra2", "lancet", "LE8_score"]  # Add LE8_score

NEGATE_SCORE_TO_RISK = True

N_POINTS = 40
Q_END = 0.95

N_BOOT = 400
RANDOM_SEED = 42
N_JOBS = -1


# ============================= utility function =============================
def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns and df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def make_times_grid(sub_df: pd.DataFrame, start_years: float, n_points: int, q_end: float):
    end_days = sub_df[time_col].quantile(q_end)
    start_days = max(30.0, start_years * 365.25)
    if end_days <= start_days:
        return None, None
    times_days = np.linspace(start_days, end_days, n_points)
    times_years = times_days / 365.25
    return times_days, times_years


# ============================= ΔiAUC + CI + P =============================
def compare_iAUC_bootstrap(sub_df: pd.DataFrame,
                          score_a: str, score_b: str,
                          start_years: float, n_points: int, q_end: float,
                          n_boot: int, seed: int, n_jobs: int,
                          negate_to_risk: bool = True):
    if sub_df.empty:
        raise ValueError("The subset was empty and iAUC could not be compared.")

    times_days, _ = make_times_grid(sub_df, start_years, n_points, q_end)
    if times_days is None:
        raise ValueError(f"There was insufficient follow-up to compare iAUC from {start_years} years.")

    n = len(sub_df)
    time_arr = sub_df[time_col].to_numpy(dtype=float)
    event_arr = sub_df[event_col].to_numpy(dtype=int).astype(bool)

    a_raw = sub_df[score_a].to_numpy(dtype=float)
    b_raw = sub_df[score_b].to_numpy(dtype=float)

    a_risk = -a_raw if negate_to_risk else a_raw
    b_risk = -b_raw if negate_to_risk else b_raw

    y_full = Surv.from_arrays(event=event_arr, time=time_arr)
    auc_a, _ = cumulative_dynamic_auc(y_full, y_full, a_risk, times_days)
    auc_b, _ = cumulative_dynamic_auc(y_full, y_full, b_risk, times_days)
    iauc_a = float(np.nanmean(auc_a))
    iauc_b = float(np.nanmean(auc_b))
    delta_hat = iauc_a - iauc_b

    rng = np.random.default_rng(seed)
    all_idx = rng.integers(0, n, size=(n_boot, n))

    def one_boot(idx):
        try:
            y_b = Surv.from_arrays(event=event_arr[idx], time=time_arr[idx])
            auc_a_b, _ = cumulative_dynamic_auc(y_b, y_b, a_risk[idx], times_days)
            auc_b_b, _ = cumulative_dynamic_auc(y_b, y_b, b_risk[idx], times_days)
            return float(np.nanmean(auc_a_b) - np.nanmean(auc_b_b))
        except Exception:
            return None

    deltas = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(one_boot)(all_idx[i]) for i in range(n_boot)
    )
    deltas = np.array([d for d in deltas if d is not None], dtype=float)

    if deltas.size < max(30, int(0.2 * n_boot)):
        raise ValueError(f"Too few bootstrap effective times（{deltas.size}/{n_boot}）。")

    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])

    p_left = np.mean(deltas <= 0.0)
    p_right = np.mean(deltas >= 0.0)
    p_value = 2.0 * min(p_left, p_right)
    p_value = float(min(max(p_value, 0.0), 1.0))

    return float(delta_hat), (float(ci_low), float(ci_high)), p_value


# ============================= main function =============================
def main():
    df = pd.read_csv(DATA_PATH, low_memory=False)

    df = coerce_numeric(df, [time_col, event_col, group_col] + scores)

    need_cols = [time_col, event_col, group_col] + scores
    df = df.dropna(subset=need_cols).copy()
    df[event_col] = df[event_col].astype(int)
    df[group_col] = df[group_col].astype(int)
    df = df[df[time_col] > 0].copy()
    
    # Automatic determination of time units: 
    # if the maximum value < 50, it is considered to be a year and needs to be converted;
    # Otherwise it is considered to be the day
    max_time = df[time_col].max()
    if max_time < 50:
        print(f"The detection time was measured in years (max={max_time:.2f}), converted to days.")
        df[time_col] = df[time_col] * 365.25
    else:
        print(f"The detection time was measured in days (max={max_time:.2f}).")

    print(f"All participants: n={len(df)}, events={int(df[event_col].sum())}")

    results_list = []

    for cohort in df['cohort'].unique():
        df_cohort = df[df['cohort'] == cohort]
        
        cohort_max_time = df_cohort['time'].max() / 365.25  # converted to years
        
        # ELSA queue special Settings
        if cohort.lower() == 'elsa':
            start_years = 5.0
            q_end = 0.70
            n_points = 20
        else:
            if cohort_max_time <= 5:
                start_years = 1.0
            elif cohort_max_time <= 10:
                start_years = 2.0
            else:
                start_years = 3.0
            q_end = Q_END
            n_points = N_POINTS

        print(f"\n{'='*60}")
        print(f"Cohort: {cohort}")
        print(f"  Max follow-up: {cohort_max_time:.2f} years")
        print(f"  Settings: start_years={start_years}, q_end={q_end}, n_points={n_points}")

        subsets = [
            ("All", df_cohort),
            ("H0", df_cohort[df_cohort[group_col] == 0]),
            ("H1", df_cohort[df_cohort[group_col] == 1]),
        ]

        for subset_name, df_sub in subsets:
            n_sample = len(df_sub)
            n_events = int(df_sub[event_col].sum())
            
            print(f"\n  [{cohort} - {subset_name}] n={n_sample}, events={n_events}")
            
            if n_sample < 50:
                print(f"    [Skip] Insufficient sample size")
                continue
            if n_events < 10:
                print(f"    [Skip] Insufficient number of events")
                continue

            for score_b in ["libra2", "lancet", "LE8_score"]:
                try:
                    delta, ci, p = compare_iAUC_bootstrap(
                        sub_df=df_sub,
                        score_a="BrainVital8", 
                        score_b=score_b,
                        start_years=start_years, 
                        n_points=n_points, 
                        q_end=q_end,
                        n_boot=N_BOOT, 
                        seed=RANDOM_SEED, 
                        n_jobs=N_JOBS,
                        negate_to_risk=NEGATE_SCORE_TO_RISK
                    )
                    
                    results_list.append({
                        "Cohort": cohort,
                        "Subset": subset_name,
                        "N": n_sample,
                        "Events": n_events,
                        "Comparison": f"BrainVital8 vs {score_b}",
                        "ΔiAUC": round(delta, 4),
                        "CI_Low": round(ci[0], 4),
                        "CI_High": round(ci[1], 4),
                        "P_value": round(p, 4)
                    })
                    
                    print(f"    BrainVital8 vs {score_b}: ΔiAUC={delta:.4f}, "
                          f"95%CI[{ci[0]:.4f}, {ci[1]:.4f}], p={p:.4g}")
                    
                except Exception as e:
                    print(f"    BrainVital8 vs {score_b}: [失败] {e}")
                    results_list.append({
                        "Cohort": cohort,
                        "Subset": subset_name,
                        "N": n_sample,
                        "Events": n_events,
                        "Comparison": f"BrainVital8 vs {score_b}",
                        "ΔiAUC": np.nan,
                        "CI_Low": np.nan,
                        "CI_High": np.nan,
                        "P_value": np.nan
                    })

    results_df = pd.DataFrame(results_list)
    output_path = CSV_DIR / "ΔiAUC_all_cohorts_comparison.csv"
    results_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    print(f"\n{'='*60}")
    print(f"✅ Done! The results have been saved to:{output_path}")
    print(f"\n Summary Table Preview:")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()
    

## AUC 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc

# ============================= Configuration =============================
PROJECT_DIR = Path("./work/")
DATA_PATH = PROJECT_DIR / "All_cohort_BASELINEHEALTH_1219.csv"

OUT_DIR = PROJECT_DIR / "Brain_Vital8_ROC" / "dynAUC_CI_and_Delta"
FIG_DIR = OUT_DIR / "figures"
CSV_DIR = OUT_DIR / "tables"
for d in [OUT_DIR, FIG_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

time_col = "time"          # days
event_col = "status"       # 1=event, 0=censored
group_col = "health_status"

scores = ["BrainVital8", "libra2", "LE8_score", "lancet"]

# Higher score indicates better health -> negate for risk direction
NEGATE_SCORE_TO_RISK = True

# Dynamic AUC parameters
N_POINTS = 40
Q_END = 0.80

START_YEARS_ALL = 3.0
START_YEARS_H0  = 3.0
START_YEARS_H1  = 3.0

# Bootstrap parallelization parameters
N_BOOT = 400
RANDOM_SEED = 42
N_JOBS = -1  # -1 = use all available cores


# ============================= utility function =============================
def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns and df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def make_times_grid(sub_df: pd.DataFrame, start_years: float, n_points: int, q_end: float):
    end_days = sub_df[time_col].quantile(q_end)
    start_days = max(30.0, start_years * 365.25)
    if end_days <= start_days:
        return None, None
    times_days = np.linspace(start_days, end_days, n_points)
    times_years = times_days / 365.25
    return times_days, times_years


# ============================= AUC + CI (bootstrap) =============================
def compute_auc_and_ci_parallel(sub_df: pd.DataFrame, score_col: str, times_days: np.ndarray,
                                n_boot: int, seed: int, n_jobs: int,
                                negate_to_risk: bool = True):
    """
    Returns: auc_t, ci_low, ci_high, mean_auc
    """
    # Full sample
    time_arr = sub_df[time_col].to_numpy(dtype=float)
    event_arr = sub_df[event_col].to_numpy(dtype=int).astype(bool)

    y_full = Surv.from_arrays(event=event_arr, time=time_arr)

    raw = sub_df[score_col].to_numpy(dtype=float)
    risk_full = -raw if negate_to_risk else raw

    auc_t, mean_auc = cumulative_dynamic_auc(y_full, y_full, risk_full, times_days)

    # Bootstrap preparation
    n = len(sub_df)
    rng = np.random.default_rng(seed)
    all_idx = rng.integers(0, n, size=(n_boot, n))

    def one_boot(idx):
        try:
            y_b = Surv.from_arrays(event=event_arr[idx], time=time_arr[idx])
            auc_b, _ = cumulative_dynamic_auc(y_b, y_b, risk_full[idx], times_days)
            return auc_b
        except Exception:
            return None

    boot_list = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(one_boot)(all_idx[i]) for i in range(n_boot)
    )
    boot_list = [b for b in boot_list if b is not None]

    if len(boot_list) < 30:
        ci_low = np.full_like(auc_t, np.nan, dtype=float)
        ci_high = np.full_like(auc_t, np.nan, dtype=float)
    else:
        boot_mat = np.asarray(boot_list)
        ci_low = np.nanpercentile(boot_mat, 2.5, axis=0)
        ci_high = np.nanpercentile(boot_mat, 97.5, axis=0)

    return auc_t, ci_low, ci_high, float(mean_auc)


# ============================= plot (AUC+CI) =============================
def dyn_auc_curves_with_ci(sub_df: pd.DataFrame, label: str,
                           start_years: float, n_points: int, q_end: float,
                           n_boot: int, seed: int, n_jobs: int,
                           pdf_path: Path, csv_path: Path,
                           negate_to_risk: bool = True):

    if sub_df.empty:
        print(f"[Skipped] {label}: subset is empty")
        return None, None

    times_days, times_years = make_times_grid(sub_df, start_years, n_points, q_end)
    if times_days is None:
        print(f"[Skipped] {label}: insufficient follow-up time from {start_years} years")
        return None, None

    results = {}

    plt.figure(figsize=(8.8, 5.6))
    for sc in scores:
        auc_t, ci_low, ci_high, mean_auc = compute_auc_and_ci_parallel(
            sub_df=sub_df, score_col=sc, times_days=times_days,
            n_boot=n_boot, seed=seed + (hash(sc) % 10000), n_jobs=n_jobs,
            negate_to_risk=negate_to_risk
        )
        results[sc] = {"auc": auc_t, "ci_low": ci_low, "ci_high": ci_high, "mean_auc": mean_auc}

        plt.plot(times_years, auc_t, linewidth=2.2, label=f"{sc} (mean={mean_auc:.3f})")
        plt.fill_between(times_years, ci_low, ci_high, alpha=0.18)

    plt.xlabel("Follow-up time (years)")
    plt.ylabel("Time-dependent AUC")
    plt.title(f"Dynamic AUC with 95% CI\n{label}")
    plt.ylim(0.0, 1.0)
    plt.xlim(start_years, times_years.max())
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close()
    print(f"📄 PDF saved: {pdf_path}")

    # Export CSV (AUC + CI)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    out = pd.DataFrame({"time_years": times_years})
    for sc in scores:
        out[f"AUC_{sc}"] = results[sc]["auc"]
        out[f"CI_low_{sc}"] = results[sc]["ci_low"]
        out[f"CI_high_{sc}"] = results[sc]["ci_high"]
    out.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"🧾 CSV saved: {csv_path}")

    return results, times_days


# ============================= main function =============================
def main():
    # Load data
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

    df = pd.read_csv(DATA_PATH, low_memory=False)
    df = coerce_numeric(df, [time_col, event_col, group_col] + scores)

    need_cols = [time_col, event_col, group_col] + scores
    df = df.dropna(subset=need_cols).copy()
    df[event_col] = df[event_col].astype(int)
    df[group_col] = df[group_col].astype(int)
    df = df[df[time_col] > 0].copy()
    
    # Convert time unit (years → days)
    df[time_col] = df[time_col] * 365.25

    print(f"All participants: n={len(df)}, events={int(df[event_col].sum())}")

    # Loop through cohorts
    for cohort in df['cohort'].unique():
        df_cohort = df[df['cohort'] == cohort]
        
        cohort_max_time = df_cohort['time'].max() / 365.25
        
        # Special settings for ELSA cohort
        if cohort.lower() == 'elsa':
            start_years = 5.0
            q_end = 0.70
            n_points = 20
            print(f"\n{'='*60}")
            print(f"Cohort: {cohort} [ELSA special settings]")
            print(f"  start_years={start_years}, q_end={q_end}, n_points={n_points}")
        else:
            # Dynamic start_years for other cohorts
            if cohort_max_time <= 5:
                start_years = 1.0
            elif cohort_max_time <= 10:
                start_years = 2.0
            else:
                start_years = 3.0
            q_end = Q_END
            n_points = N_POINTS
            print(f"\n{'='*60}")
            print(f"Cohort: {cohort}")
        
        print(f"  Max follow-up: {cohort_max_time:.2f} years, start_years={start_years}")

        # Define 3 subsets: All, H0, H1
        subsets = [
            ("All", df_cohort),
            ("H0", df_cohort[df_cohort[group_col] == 0]),
            ("H1", df_cohort[df_cohort[group_col] == 1]),
        ]

        for subset_name, df_sub in subsets:
            n_sample = len(df_sub)
            n_events = int(df_sub[event_col].sum())
            
            print(f"\n  [{cohort} - {subset_name}] n={n_sample}, events={n_events}")
            
            # Check sample size and event count
            if n_sample < 50:
                print(f"    [Skipped] Insufficient sample size (n={n_sample} < 50)")
                continue
            if n_events < 10:
                print(f"    [Skipped] Insufficient events (events={n_events} < 10)")
                continue

            dyn_auc_curves_with_ci(
                sub_df=df_sub,
                label=f"{cohort} - {subset_name} (n={n_sample}, events={n_events})",
                start_years=start_years, 
                n_points=n_points,  # Use variable instead of N_POINTS
                q_end=q_end,        # Use variable instead of Q_END
                n_boot=N_BOOT, 
                seed=RANDOM_SEED, 
                n_jobs=N_JOBS,
                pdf_path=FIG_DIR / f"{cohort}_{subset_name}_dynamic_auc.pdf",
                csv_path=CSV_DIR / f"{cohort}_{subset_name}_dynamic_auc.csv",
                negate_to_risk=NEGATE_SCORE_TO_RISK
            )

    print("\n" + "="*60)
    print("✅ Completed! Generated 3 PDF and CSV files per cohort:")
    print("   - {cohort}_All_dynamic_auc.pdf")
    print("   - {cohort}_H0_dynamic_auc.pdf") 
    print("   - {cohort}_H1_dynamic_auc.pdf")


if __name__ == "__main__":
    main()